In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn_pandas import DataFrameMapper
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn import metrics

# URLs de tu repositorio de GitHub (formato Raw)
url_test = 'https://raw.githubusercontent.com/LucasAimetta/grupo3_aimetta_heinen_ia_2026/refs/heads/main/Competencia%20ML/solicitudes_test.csv'
url_comportamiento = 'https://raw.githubusercontent.com/LucasAimetta/grupo3_aimetta_heinen_ia_2026/refs/heads/main/Competencia%20ML/comportamiento_bancario.csv'
url_train = 'https://raw.githubusercontent.com/LucasAimetta/grupo3_aimetta_heinen_ia_2026/refs/heads/main/Competencia%20ML/solicitudes_train.csv'

# Cargar los datos directamente desde GitHub
df_solicitudes_test = pd.read_csv(url_test)
df_comportamiento = pd.read_csv(url_comportamiento)
df_solicitudes_train = pd.read_csv(url_train)

In [12]:

df_comportamiento['Mes_Referencia'] = pd.to_datetime(df_comportamiento['Mes_Referencia'])
def agregar_comportamiento(df_solicitudes_base, df_comportamiento):
    df_temp = pd.merge(
        left=df_solicitudes_base[['ID_Solicitud','ID_Usuario','Fecha_Solicitud']],
        right=df_comportamiento[['ID_Usuario','Mes_Referencia','Saldo_Promedio_Mensual','Dias_Atraso_Otras_Deudas']],
        on='ID_Usuario',
        how='left'
    )
    # Agrupamos por el ID_Usuario y hacemos promedio de las columnas saldo y dias atraso
    df_temp = df_temp[df_temp['Mes_Referencia'] < df_temp['Fecha_Solicitud']]
    return df_temp.groupby('ID_Solicitud')[['Saldo_Promedio_Mensual','Dias_Atraso_Otras_Deudas']].mean()


In [13]:

# Para train:
df_comportamiento_agrupado_train = agregar_comportamiento(df_solicitudes_train, df_comportamiento)
df_cruzado = pd.merge(df_solicitudes_train, df_comportamiento_agrupado_train, on='ID_Solicitud', how='left')
df_cruzado.sort_values(by='ID_Usuario', ascending=True)

# Para test:
df_comportamiento_agrupado_test = agregar_comportamiento(df_solicitudes_test, df_comportamiento)
df_solicitudes_test_cruzado = pd.merge(df_solicitudes_test, df_comportamiento_agrupado_test, on='ID_Solicitud', how='left')

#Creamos columnas Monto a Pagar en cada set
df_cruzado['Monto_A_Pagar'] = df_cruzado['Monto_Solicitado'] * df_cruzado['Tasa_Interes']
df_solicitudes_test_cruzado['Monto_A_Pagar'] = df_solicitudes_test_cruzado['Monto_Solicitado'] * df_solicitudes_test_cruzado['Tasa_Interes']


#Separamos del set de train, un set de train y uno de validation
train,validation = train_test_split(df_cruzado, test_size = 0.3, random_state = 70, stratify=df_cruzado['Estado_Final'])


In [14]:



mapper = DataFrameMapper([
(['Monto_A_Pagar'],[SimpleImputer(strategy='mean'), StandardScaler()]),
(['Ingreso_Mensual'],[SimpleImputer(strategy='mean'), StandardScaler()]),
(['Saldo_Promedio_Mensual'],[SimpleImputer(strategy='mean'), StandardScaler()]),
(['Dias_Atraso_Otras_Deudas'],[SimpleImputer(strategy='mean'), StandardScaler()])
])
#Seleccionamos que busque los 3 vecinos mas cercanos.
K = 3

knn_model = Pipeline([
    ('mapper', mapper),
    ('classifier', KNeighborsClassifier(n_neighbors=K))
])
#Entrenamos con train
knn_model.fit(train, train['Estado_Final'])
knn_model.predict(validation)




/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:95: FutureWarning: Function tosequence is deprecated; tosequence was deprecated in 1.5 and will be removed in 1.7
  warnings.warn(msg, category=FutureWarning)


array([1, 0, 0, ..., 1, 1, 1])

In [15]:

df_solicitudes_test_cruzado['Ganancia']= (df_solicitudes_test_cruzado['Monto_Solicitado'])*(df_solicitudes_test_cruzado['Tasa_Interes'])
monto_solicitado_total = 0
df_solicitudes_test_cruzado['Prob_Pago'] = knn_model.predict(df_solicitudes_test_cruzado)

df_solicitudes_test_cruzado['Estado'] =0


In [16]:
#CREAMOS LISTA DE INDICES DE LAS FILAS DE LAS SOLICITUDES QUE APROBAREMOS
approved_indices = []
ganancia_esperada = 0
#ITERAMOS CADA FILA
for index, row in df_solicitudes_test_cruzado.sort_values(by='Ganancia', ascending=False).iterrows():
    if ((monto_solicitado_total + row['Monto_Solicitado']) < 2000000) and (row['Prob_Pago']==1):
        monto_solicitado_total += row['Monto_Solicitado']
        ganancia_esperada += row['Ganancia']

        approved_indices.append(index)



# ACTUALIZAMOS EL ESTADO A 1, DE DICHAS SOLICITUDES QUE APROBAMOS
df_solicitudes_test_cruzado.loc[approved_indices, 'Estado'] = 1
df_solicitudes_test_cruzado
df_solicitudes_test_cruzado['Estado'].value_counts(),monto_solicitado_total,ganancia_esperada

(Estado
 0    2775
 1     286
 Name: count, dtype: int64,
 1999800,
 594584.2399999995)

In [17]:
df_resultados = df_solicitudes_test_cruzado[['ID_Solicitud', 'Estado']].copy()
df_resultados.rename(columns={'Estado': 'Decision_Aprobacion'}, inplace=True)

# Exportar a CSV
df_resultados.to_csv('decisiones_aprobacion.csv', index=False)

print('Archivo decisiones_aprobacion.csv creado exitosamente.')
display(df_resultados.head())

Archivo decisiones_aprobacion.csv creado exitosamente.


,ID_Solicitud,Decision_Aprobacion
0,32417,0
1,32418,1
2,32419,0
3,32420,0
4,32421,0
